<a href="https://colab.research.google.com/github/jefc21/Estrutura-de-Dados---Algoritmos-de-Ordena-o-/blob/main/Atividade_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#LEITURA DAS INSTÂNCIAS E CONFIGURAÇÃO DAS ENTRADAS PARA OS ALGORITMOS

In [26]:
pai = {}
import urllib.request
#link da instancia dij10.txt -> https://drive.google.com/file/d/1rIZ8dHqgydxWKXJHM5xTYwgult7EKmzx/view?usp=sharing
#link da pasta das instancias -> https://drive.google.com/drive/folders/18Qaag-bHipC0oD5KAqhhycEJF74PVKK8?usp=sharing
def load(instancia):
  # IDs dos arquivos de instancias na pasta do drive
  arquivos = {
    "dij10.txt" : "1rIZ8dHqgydxWKXJHM5xTYwgult7EKmzx",
    "dij20.txt" : "1LLjd5_kaTJBwLSfvunf5EehBrIluyJMM",
    "dij40.txt" : "1NLZRRitThDXPLXZbOB2Fml1WNeuXfFvw",
    "dij50.txt" : "1HaCAj0LF00utuhFRL7mdrWqLxNhuCEoY",
  }

  file_id = arquivos[instancia]
  url = f"https://drive.google.com/uc?export=download&id={file_id}"
  urllib.request.urlretrieve(
    url,
    instancia
  )

  # carregar as instancias e configurar conforme a entrada do algortimo
  with open(instancia, "r", encoding="utf-8") as arquivo:
      linhas = arquivo.readlines()
      n = int(linhas[0])
  vertices = []
  arestas = []
  G = {}
  # Configuração de Entrada para Kruskal
  for i in range(0, n-1):
      vertices.append(i)
      G[i] = []
      for j in range(0, n-i-1):
          arestas.append((i, j+i+1, int(linhas[i+1].replace("\t"," ").split(' ')[j])))
  vertices.append(n-1)

  # Configuração de Entrada para Prim
  G[n-1]=[]
  for x in arestas:
      G.get(x[0],0).append((x[1],x[2]))
      G.get(x[1],0).append((x[0],x[2]))

  return arestas, vertices, G

In [ ]:
#FUNÇÕES/METODOS "AUXILIARES"

In [27]:
def merge_sort(array):
    if len(array) > 1:
        middle =  len(array) // 2
        array_left= array[:middle]
        array_right = array[middle:]

        merge_sort(array_left)
        merge_sort(array_right)

        i=0
        j=0
        k=0

        while i < len(array_left) and j < len(array_right):
            if array_left[i][2] < array_right[j][2]:
                array[k] = array_left[i]
                i += 1
            else:
                array[k] = array_right[j]
                j += 1
            k += 1

        while i < len(array_left):
            array[k] = array_left[i]
            i += 1
            k += 1

        while j < len(array_right):
            array[k] = array_right[j]
            j += 1
            k += 1
    return array

#-------------------------------------------------
def makeSet(v):
    pai[v] = v

#-------------------------------------------------

def findSet(v):
    if pai[v] == v:
        return v
    return findSet(pai[v])

#------------------------------------------------
def union(u, v):
        pai[findSet(u)] = findSet(v)

#-------------------------------------------------

def minHeapify(A, chave, i):
    L = 2 * i + 1
    R = 2 * i + 2
    tamanho_heap = len(A)
    menor = i

    if L < tamanho_heap and chave[A[L]] < chave[A[menor]]:
        menor = L
    if R < tamanho_heap and chave[A[R]] < chave[A[menor]]:
        menor = R
    if menor != i:
        [A[i]], [A[menor]] = [A[menor]], [A[i]]
        minHeapify(A, chave, menor)


def extractMin(Q, chave):

    buildMinHeap(Q, chave)

    u = Q[0]       # menor elemento
    Q[0] = Q[-1]
    Q.pop()
    minHeapify(Q, chave, 0,)  # reorganiza heap

    return u


def buildMinHeap(A, chave):
    tamanho_heap = len(A)
    for i in range((tamanho_heap//2)-1,-1, -1):
        minHeapify(A, chave, i)



In [ ]:
#ALGORITMO DE KRUSKAL

In [28]:
def kruskal(vertices, arestas):
    peso_total = 0
    arvore = []
    pai = {}

    # ordenação dos vértices pelo peso
    merge_sort(arestas)

    # inicializa cada vértice como sendo pai de se mesmo
    for vertice in vertices:
        makeSet(vertice)

    for aresta in arestas:
        if findSet(aresta[0]) != findSet(aresta[1]):
            arvore.append((aresta[0], aresta[1], aresta[2]))
            peso_total += aresta[2]

            union(aresta[0], aresta[1])
    return arvore, peso_total

#print(kruskal(vertices, arestas))

In [ ]:
#ALGORITMO DE PRIM

In [29]:
def prim(G, r):
    pai = {}
    chave = {}

    for u in G:
        pai[u] = None
        chave[u] = float('inf')
    chave[r] = 0
    Q = list(G.keys())
    buildMinHeap(Q, chave)

    while Q:
        u = extractMin(Q, chave)
        for v, peso in G[u]:
            if ((v in Q) and (peso < chave[v])):
                pai[v] = u
                chave[v] = peso
                buildMinHeap(Q, chave)

    peso_total = 0
    for v in pai:
        if pai[v] is not None:
            for vizinho, peso in G[v]:
                if vizinho == pai[v]:
                    peso_total += peso
    return pai, peso_total

#result = prim(G,1)
#print(result)


In [ ]:
#ALGORITMO DE DIJKSTRA

In [30]:
def dijkstra(G, r):
    pai = {}
    dist = {}

    for u in G:
        pai[u] = None
        dist[u] = float('inf')
    dist[r] = 0
    Q = list(G.keys())
    buildMinHeap(Q, dist)

    while Q:
        u = extractMin(Q, dist)
        for v, peso in G[u]:
            if ((v in Q) and (dist[u] + peso < dist[v])):
                pai[v] = u
                dist[v] = dist[u] + peso
                buildMinHeap(Q, dist)

    return pai, dist

#result = dijkstra(G,1)
#print(result)


In [35]:
############################################################################################################
# Importante: Execute as células anteriores antes de rodar este bloco para carregar as funções necessárias.#
# instancias: dij10.txt,  dij20.txt, dij40.txt, dij50.txt                                                                                                         #
############################################################################################################

def main():
  arestas, vertices, G = load("dij10.txt")
  kruskal_ = kruskal(vertices, arestas)
  prim_ = (prim(G,1))
  dijkstra_ = dijkstra(G,0)

  print("Algoritmo KRUSKAL \n de Grafo: ", kruskal_[0], " - Solução: ", kruskal_[1], "\n \n ")
  print("Algoritmo PRIM \n Grafo: ",prim_[0], " - Solução: ", prim_[1], "\n \n")
  print("Algoritmo DIJKSTRA \n Grafo: ",dijkstra_[0], " - Solução:", dijkstra_[1])

main()

Algoritmo KRUSKAL 
 de Grafo:  [(28, 29, 25), (33, 34, 86), (13, 30, 107), (1, 3, 259), (11, 21, 299), (11, 20, 313), (3, 4, 333), (15, 17, 341), (5, 6, 353), (30, 31, 384), (6, 7, 391), (24, 25, 396), (43, 46, 413), (44, 45, 441), (47, 49, 456), (7, 8, 458), (18, 19, 474), (26, 27, 480), (42, 43, 493), (8, 9, 504), (17, 19, 526), (3, 5, 555), (19, 22, 578), (15, 21, 584), (38, 41, 594), (23, 26, 595), (2, 4, 607), (11, 14, 611), (46, 48, 648), (38, 40, 648), (12, 31, 657), (36, 39, 665), (14, 16, 669), (22, 23, 676), (41, 42, 683), (25, 28, 685), (35, 40, 710), (37, 39, 723), (23, 28, 833), (30, 33, 911), (0, 11, 955), (10, 12, 985), (34, 36, 1008), (12, 32, 1080), (7, 24, 1121), (48, 49, 1137), (14, 30, 1218), (44, 47, 1330), (36, 38, 1426)]  - Solução:  30424 
 
 
Algoritmo PRIM 
 Grafo:  {0: 11, 1: None, 2: 4, 3: 1, 4: 3, 5: 3, 6: 5, 7: 6, 8: 7, 9: 8, 10: 12, 11: 21, 12: 31, 13: 30, 14: 11, 15: 17, 16: 14, 17: 19, 18: 19, 19: 22, 20: 11, 21: 15, 22: 23, 23: 28, 24: 7, 25: 24, 26: 2